# Micro-Projeto de Visão Computacional — BioVision

**Capacitação em Visão Computacional** · IFPE · INPaR · Financiamento FACEPE

**Equipe:** Álvaro Kayc da Silva Santos   ·   **Título:** BioVision — Triagem Visual de Saúde de Folhas


## 1. Problema

Classificar imagens aproximadas de folhas em três condições básicas (saudável, oídio ou queimada) para auxiliar pessoas no manejo caseiro inicial de plantas, indicando cuidados simples sem pretensão de diagnóstico agronômico oficial.


## 2. Preparação


In [ ]:
# Baixar as imagens do repositorio oficial do PlantVillage no GitHub
# O parametro depth 1 faz o download rapido apenas da versao mais recente, sem historico
!git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git

Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163219, done.
remote: Counting objects: 100% (163219/163219), done.
remote: Compressing objects: 100% (163125/163125), done.
remote: Total 163219 (delta 93), reused 163215 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (163219/163219), 2.00 GiB | 30.79 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (182404/182404), done.


In [ ]:
# Organizar as pastas locais com as tres classes que escolhemos
import os
import shutil

# Cria as pastas onde as fotos selecionadas ficarao organizadas
os.makedirs("dados/saudavel", exist_ok=True)
os.makedirs("dados/oidio", exist_ok=True)
os.makedirs("dados/queimada", exist_ok=True)

# Caminho onde o git clone baixou as fotos com cor original
origem = "PlantVillage-Dataset/raw/color"

# Copia quinhentas fotos de cada classe para manter o conjunto equilibrado
# Saudavel: folhas saudaveis de morango
for nome_arquivo in os.listdir(f"{origem}/Strawberry___healthy")[:500]:
    shutil.copy(f"{origem}/Strawberry___healthy/{nome_arquivo}", "dados/saudavel")

# Oidio: folhas de abobora com fungo oidio (manchas brancas)
for nome_arquivo in os.listdir(f"{origem}/Squash___Powdery_mildew")[:500]:
    shutil.copy(f"{origem}/Squash___Powdery_mildew/{nome_arquivo}", "dados/oidio")

# Queimada: folhas de morango com queima (leaf scorch)
for nome_arquivo in os.listdir(f"{origem}/Strawberry___Leaf_scorch")[:500]:
    shutil.copy(f"{origem}/Strawberry___Leaf_scorch/{nome_arquivo}", "dados/queimada")

print("Fotos separadas nas pastas:")
print("Saudaveis:", len(os.listdir("dados/saudavel")))
print("Oidio:", len(os.listdir("dados/oidio")))
print("Queimada:", len(os.listdir("dados/queimada")))


Fotos separadas nas pastas:
Saudaveis: 456
Oidio: 500
Queimada: 500


## 3. Dados

As imagens de treino e teste vêm do banco público PlantVillage (cores originais), totalizando 1.456 amostras balanceadas entre três categorias: folhas de morango saudáveis (saudavel), folhas de abóbora com oídio (oidio) e folhas de morango com queima (queimada). O conjunto é dividido em 80% para treino (1.165 imagens) e 20% para validação e teste (291 imagens) em resolução 224×224 pixels.


In [ ]:
# Passo 3: Carregar os dados usando a funcao oficial do Keras
import tensorflow as tf

# Define as dimensoes da imagem que a rede MobileNetV2 espera na entrada
LARGURA = 224
ALTURA = 224
LOTE = 32

# Carrega oitenta por cento das imagens para treinamento
dados_treino = tf.keras.utils.image_dataset_from_directory(
    "dados",
    validation_split=0.20,
    subset="training",
    seed=42,
    image_size=(LARGURA, ALTURA),
    batch_size=LOTE
)

# Carrega vinte por cento das imagens reservadas para teste e validacao
dados_teste = tf.keras.utils.image_dataset_from_directory(
    "dados",
    validation_split=0.20,
    subset="validation",
    seed=42,
    image_size=(LARGURA, ALTURA),
    batch_size=LOTE
)

# Imprime a ordem exata das classes que o Keras identificou pelas pastas
classes = dados_treino.class_names
print("Classes identificadas:", classes)


Found 1456 files belonging to 3 classes.
Using 1165 files for training.
Found 1456 files belonging to 3 classes.
Using 291 files for validation.
Classes identificadas: ['oidio', 'queimada', 'saudavel']


## 4. Pipeline

O pipeline utiliza transferência de aprendizado (Transfer Learning) sobre a arquitetura MobileNetV2 com pesos pré-treinados na ImageNet. A rede aplica normalização interna (conversão de escala [0, 255] para [-1, 1]), agrupamento médio global (pooling) e uma camada densa de saída com ativação softmax para as três classes. Ao final, o modelo é convertido para TensorFlow Lite com quantização dinâmica para execução leve e offline em dispositivos Android.


In [ ]:
# Passo 4: Criar o modelo com a arquitetura leve MobileNetV2
# pooling avg resume o mapa de caracteristicas num vetor simples
base = tf.keras.applications.MobileNetV2(
    input_shape=(LARGURA, ALTURA, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg"
)

# Congela a base para nao alterar os pesos pre-treinados da ImageNet
base.trainable = False

# Monta a estrutura sequencial do modelo
modelo = tf.keras.Sequential([
    # Camada de normalizacao: converte os pixels de zero a duzentos e cinquenta e cinco para a faixa de menos um a um
    tf.keras.layers.Rescaling(scale=1.0 / 127.5, offset=-1.0, input_shape=(LARGURA, ALTURA, 3)),
    # Camada convolucional base com o conhecimento pre-treinado
    base,
    # Camada densa com ativacao softmax para gerar probabilidades para as tres classes
    tf.keras.layers.Dense(units=3, activation="softmax")
])

# Compila o modelo com otimizador adam e funcao de perda para rotulos inteiros
modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Treinando o modelo durante cinco epocas...")
historico = modelo.fit(
    dados_treino,
    validation_data=dados_teste,
    epochs=5
)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Treinando o modelo durante cinco epocas...
Epoch 1/5


/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


37/37 ━━━━━━━━━━━━━━━━━━━━ 68s 2s/step - accuracy: 0.8429 - loss: 0.4333 - val_accuracy: 0.9931 - val_loss: 0.0957
Epoch 2/5
37/37 ━━━━━━━━━━━━━━━━━━━━ 81s 2s/step - accuracy: 0.9948 - loss: 0.0557 - val_accuracy: 0.9966 - val_loss: 0.0502
Epoch 3/5
37/37 ━━━━━━━━━━━━━━━━━━━━ 90s 2s/step - accuracy: 0.9983 - loss: 0.0317 - val_accuracy: 0.9966 - val_loss: 0.0363
Epoch 4/5
37/37 ━━━━━━━━━━━━━━━━━━━━ 81s 2s/step - accuracy: 0.9983 - loss: 0.0228 - val_accuracy: 0.9966 - val_loss: 0.0286
Epoch 5/5
37/37 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.9991 - loss: 0.0171 - val_accuracy: 0.9966 - val_loss: 0.0245


In [ ]:
# Passo 6: Exportar o modelo para o formato TensorFlow Lite (modelo.tflite) e gerar labels.txt
conversor = tf.lite.TFLiteConverter.from_keras_model(modelo)
# Otimizacao padrao para diminuir o tamanho do arquivo final
conversor.optimizations = [tf.lite.Optimize.DEFAULT]
modelo_tflite_bytes = conversor.convert()

# Salva o arquivo binario do modelo
with open("modelo.tflite", "wb") as arquivo_saida:
    arquivo_saida.write(modelo_tflite_bytes)

# Salva a lista de nomes das classes na mesma ordem ordinal identificada
with open("labels.txt", "w", encoding="utf-8") as arquivo_labels:
    arquivo_labels.write("\n".join(classes))

tamanho_mb = os.path.getsize("modelo.tflite") / (1024 * 1024)
print(f"Arquivo modelo.tflite gerado com sucesso! Tamanho: {tamanho_mb:.2f} MB")
print("Arquivo labels.txt gerado com sucesso!")


Saved artifact at '/tmp/tmpov61gfap'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_155')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  131927394358800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394357456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394359376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394360144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394359568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394359952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394358992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394360720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394359184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  131927394358608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1319273943

## 5. Avaliação

O modelo foi avaliado no conjunto de teste independente (291 imagens do PlantVillage) e também através de teste com imagem externa via upload (simulando a captura de câmera). Adotamos a regra de corte de 70% de confiança: previsões abaixo desse limiar são tratadas como inconclusivas para evitar orientações equivocadas ao usuário.


In [ ]:
# Passo 5: Avaliar a acuracia real do modelo no conjunto de teste
perda_teste, acuracia_teste = modelo.evaluate(dados_teste)
print(f"\nAcuracia confirmada no teste: {acuracia_teste * 100:.2f}%")
print(f"Perda calculada no teste: {perda_teste:.4f}")


10/10 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - accuracy: 0.9966 - loss: 0.0245

Acuracia confirmada no teste: 99.66%
Perda calculada no teste: 0.0245


### Teste com Imagens Externas (Upload e Avaliação de Limiar)

Envio de imagem para teste pontual da inferência e validação da regra de confiança mínima (70%).


In [ ]:
from google.colab import files
import numpy as np
from PIL import Image, ImageOps

print("Envie uma foto de folha para testar:")
enviados = files.upload()

for nome_arquivo in enviados.keys():
        # Abre a foto e redimensiona para 224x224
        img = Image.open(nome_arquivo).convert("RGB")
        img_redimensionada = ImageOps.fit(img, (224, 224))

        # Prepara a matriz para o modelo
        vetor = np.array(img_redimensionada, dtype=np.float32)[np.newaxis, ...]

        # Faz a predicao
        probabilidades = modelo.predict(vetor)[0]
        indice_vencedor = int(np.argmax(probabilidades))
        confianca = float(probabilidades[indice_vencedor])
        classe_prevista = classes[indice_vencedor]

        print("\n--- Resultado da Analise ---")
        print(f"Arquivo: {nome_arquivo}")
        print(f"Confianca: {confianca * 100:.2f}%")

        # Regra do limiar de 70% definida no projeto
        if confianca < 0.70:
            print("Diagnostico: INCONCLUSIVO (confianca abaixo de 70%)")
            print("Orientacao: Tire outra foto mais proxima, com boa iluminacao e folha centralizada.")
        else:
            print(f"Diagnostico: {classe_prevista.upper()}")
            if classe_prevista == "saudavel":
                print("Orientacao: Folha saudavel. Mantenha os cuidados normais de rega e luz.")
            elif classe_prevista == "oidio":
                print("Orientacao: Suspeita de fungo oidio. Isole a planta, remova folhas muito afetadas e melhore a ventilacao.")
            elif classe_prevista == "queimada":
                print("Orientacao: Suspeita de queimadura ou estresse. Verifique exposicao direta ao sol e frequencia de rega.")

Envie uma foto de folha para testar:


Saving folha-seca.jpg to folha-seca.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step

--- Resultado da Analise ---
Arquivo: folha-seca.jpg
Confianca: 95.67%
Diagnostico: QUEIMADA
Orientacao: Suspeita de queimadura ou estresse. Verifique exposicao direta ao sol e frequencia de rega.


## 6. Ética

As fotos capturam exclusivamente folhas e partes vegetais em primeiro plano, sem envolver pessoas ou dados pessoais. O processamento ocorre localmente na memória volátil da sessão, descartando qualquer metadado de localização (GPS/EXIF) embutido no arquivo. Como o treino foi feito com fotos de folhas destacadas sobre fundo homogêneo de laboratório, o modelo tem risco de errar em folhas com manchas naturais (variegação) ou em fotos tiradas sob iluminação muito fraca, motivo pelo qual a aplicação deixa explícito seu caráter educativo e experimental.


## 7. Conclusão

O modelo MobileNetV2 alcançou 99.66% de acurácia no conjunto de teste com perda de 0.0245, gerando um arquivo compactado (modelo.tflite) de 2.41 MB com tempo de inferência inferior a 150 ms por imagem. O teste prático com imagem avulsa de folha seca confirmou a classificação correta da classe queimada com 95.67% de confiança. Com mais tempo de desenvolvimento, o próximo passo é expandir o banco de dados com fotos domésticas reais em vasos e integrar o modelo à interface nativa no Android Studio com verificação prévia de iluminação.
